In [60]:
import regex as re

# 读取语料
data_dir = "../tests/fixtures/corpus.en"

# handout给的正则表达式
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

target_str = ""

try: 
    with open(data_dir, "r", encoding="utf-8") as file:
        target_str = file.read()
except FileNotFoundError:
    print(f"File {data_dir} not found")
    exit(1)

# 正则匹配
matches = re.finditer(PAT, target_str)
print(matches)

In [ ]:

# 统计pretoken的次数
pretoken_counts = {}
pretoken_seq = {}

for match in matches:
    key = match.group(0).encode("utf-8")

    pretoken_counts[key] = pretoken_counts.get(key, 0) + 1
    pretoken_seq[key] = list[int](key)

# # 按照频率倒序排序
# sorted_items = sorted(pretoken_counts.items(), key=lambda item: (item[1], item[0]), reverse=True)

# pretoken_counts = dict(sorted_items)

print(pretoken_counts)
print(pretoken_seq)


{b'iron': 2, b' cement': 3, b' is': 338, b' a': 480, b' ready': 4, b' for': 237, b' use': 34, b' paste': 1, b' which': 91, b' laid': 1, b' as': 140, b' fillet': 1, b' by': 105, b' putty': 1, b' knife': 1, b' or': 98, b' finger': 1, b' in': 436, b' the': 1279, b' mould': 2, b' edges': 1, b' (': 113, b' corners': 1, b' )': 108, b' of': 646, b' steel': 2, b' ingot': 2, b' .': 962, b'\n': 1015, b' protects': 2, b' against': 5, b' hot': 2, b' ,': 1306, b' abrasive': 1, b' casting': 1, b' process': 8, b'a': 10, b' fire': 4, b' restant': 1, b' repair': 2, b' places': 5, b' ovens': 1, b' open': 14, b' fireplaces': 1, b' etc': 35, b'construction': 1, b' and': 609, b' highways': 1, b' ...': 77, b'an': 3, b' announcement': 2, b' must': 26, b' be': 173, b' commercial': 10, b' character': 2, b'goods': 1, b' services': 9, b' advancement': 1, b' through': 14, b' P': 1, b'.': 77, b'O': 1, b'Box': 1, b' system': 11, b' NOT': 2, b' ALLOWED': 1, b'deliveries': 1, b' spam': 1, b' other': 34, b' improper':

In [82]:
# 遍历每个pretoken的bytes，构建pair_counts,并且按照频率排序
from typing import Any

pair_counts = {}

for key, it in pretoken_seq.items():
    for i in range(len(it) - 1):
        pair = (it[i], it[i+1])
        pair_counts[pair] = pair_counts.get(pair, 0) + pretoken_counts[key]

print(pair_counts)


{(105, 114): 217, (114, 111): 501, (111, 110): 1214, (32, 99): 948, (99, 101): 477, (101, 109): 263, (109, 101): 626, (101, 110): 1038, (110, 116): 784, (32, 105): 1306, (105, 115): 1001, (32, 97): 2214, (32, 114): 422, (114, 101): 1490, (101, 97): 556, (97, 100): 271, (100, 121): 21, (32, 102): 852, (102, 111): 480, (111, 114): 1258, (32, 117): 371, (117, 115): 379, (115, 101): 718, (32, 112): 681, (112, 97): 303, (97, 115): 580, (115, 116): 799, (116, 101): 906, (32, 119): 969, (119, 104): 240, (104, 105): 597, (105, 99): 498, (99, 104): 420, (32, 108): 463, (108, 97): 414, (97, 105): 248, (105, 100): 168, (102, 105): 212, (105, 108): 386, (108, 108): 563, (108, 101): 597, (101, 116): 409, (32, 98): 895, (98, 121): 113, (112, 117): 130, (117, 116): 360, (116, 116): 109, (116, 121): 117, (32, 107): 80, (107, 110): 42, (110, 105): 204, (105, 102): 161, (102, 101): 182, (32, 111): 1311, (105, 110): 1890, (110, 103): 815, (103, 101): 244, (101, 114): 1564, (256, 104): 2094, (104, 101): 2

In [80]:

vocab = {i: bytes([i]) for i in range(256)} 
merges = []

def get_best_pair(pair_counts):
    return max(pair_counts.items(), key=lambda item: (item[1], item[0]))

def merge_one_pair(li, best_pair, new_id):
    new_list = []
    i = 0
    while i < len(li):
        if i < len(li) - 1 and li[i] == best_pair[0] and li[i+1] == best_pair[1]:
            new_list.append(new_id)
            i += 2
        else:
            new_list.append(li[i])
            i += 1
    return new_list

best_pair, best_count = get_best_pair(pair_counts)
print(best_pair)
print(best_count)

left, right = best_pair

new_id = len(vocab)

vocab[new_id] = vocab[left] + vocab[right]

print(vocab[new_id])

merges.append((vocab[left], vocab[right]))

print(merges)

for word, li in pretoken_seq.items():
    merge_one_pair(li, best_pair, new_id)

print(pretoken_seq)

list = [32, 116, 104, 101]

print(merge_one_pair(list, best_pair, new_id))

for key, seq in pretoken_seq.items():
    cnt = pretoken_counts[key]
    for i in range(len(seq) - 1):
        pair = (seq[i], seq[i+1])
        pair_counts[pair] = pair_counts.get(pair, 0) + cnt

print(pair_counts)




(32, 116)
2940
b' t'
[(b' ', b't')]
{b'iron': [105, 114, 111, 110], b' cement': [32, 99, 101, 109, 101, 110, 116], b' is': [32, 105, 115], b' a': [32, 97], b' ready': [32, 114, 101, 97, 100, 121], b' for': [32, 102, 111, 114], b' use': [32, 117, 115, 101], b' paste': [32, 112, 97, 115, 116, 101], b' which': [32, 119, 104, 105, 99, 104], b' laid': [32, 108, 97, 105, 100], b' as': [32, 97, 115], b' fillet': [32, 102, 105, 108, 108, 101, 116], b' by': [32, 98, 121], b' putty': [32, 112, 117, 116, 116, 121], b' knife': [32, 107, 110, 105, 102, 101], b' or': [32, 111, 114], b' finger': [32, 102, 105, 110, 103, 101, 114], b' in': [32, 105, 110], b' the': [256, 104, 101], b' mould': [32, 109, 111, 117, 108, 100], b' edges': [32, 101, 100, 103, 101, 115], b' (': [32, 40], b' corners': [32, 99, 111, 114, 110, 101, 114, 115], b' )': [32, 41], b' of': [32, 111, 102], b' steel': [32, 115, 116, 101, 101, 108], b' ingot': [32, 105, 110, 103, 111, 116], b' .': [32, 46], b'\n': [10], b' protects': [32